# Library Inclusion and Utilities

<!-- structured-notebook -->
## Notebook Summary
Purpose: build a persistent Chroma vector database for ProQuest news articles using precomputed embeddings so later notebooks can retrieve supporting examples during LLM-based classification.

Main steps:
- Load the cleaned article texts and the matching precomputed embedding matrix.
- Check that the text and embedding counts still line up after filtering empty documents.
- Create or reuse a Chroma collection and index the documents in batches.


In [ ]:
# structured-notebook-bootstrap
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


_repo_root = find_repo_root(Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.append(str(_repo_root))

from src.project_paths import (
    ARXIV_RAW_DIR,
    CHROMA_DIR,
    EXTERNAL_NEWS_DIR,
    GUARDIAN_DATA_DIR,
    LLM_CLASSIFICATION_DIR,
    NEWS_HTML_DIR,
    NEWS_OUTPUT_DIR,
    PREPRINT_RAW_DIR,
    PROQUEST_PROCESSED_DIR,
    PROQUEST_UNPROCESSED_DIR,
    PUBMED_PROCESSED_DIR,
    PUBMED_RAW_DIR,
    PUBLICATIONS_TABLE_DIR,
    REDDIT_DATA_DIR,
    ROOT,
    RQ1_FIGURES_DIR,
    RQ4_PLOTS_DIR,
    TOPIC_MATCHING_DIR,
    YOUTUBE_DATA_DIR,
)


In [1]:
import pandas as pd
import numpy as np
import re

In [8]:
df = pd.read_csv(PROQUEST_PROCESSED_DIR / 'full_df.csv')

<!-- structured-notebook -->
## Embedding Preparation
This block loads the precomputed embedding matrix and ensures it still lines up with the filtered document list before anything is written to Chroma.


In [7]:
#Loading pre-computed embeddings
X = np.load(PROQUEST_PROCESSED_DIR / 'final_embed.npy')

In [9]:
mask = df["Full text"].fillna("").str.strip() != ""
docs = df.loc[mask, "Full text"].astype(str).tolist()

In [10]:
assert len(docs) == X.shape[0], f"docs={len(docs)} but embeds={X.shape[0]}"

In [14]:
ids = [f"doc_{i}" for i in range(len(docs))]

<!-- structured-notebook -->
## Chroma Ingestion
The final cells create the persistent collection and add documents in batches so the vector store can be reused by the LangGraph notebook.


In [17]:
import chromadb

chroma_path = CHROMA_DIR   # folder will be created
client = chromadb.PersistentClient(path=chroma_path)
collection = client.get_or_create_collection(name="news_articles")

In [18]:
batch_size = 500

for start in range(0, len(docs), batch_size):
    end = min(start + batch_size, len(docs))

    collection.add(
        ids=ids[start:end],
        documents=docs[start:end],
        embeddings=X[start:end].tolist()
    )
    
    print(f"Indexed {end}/{len(docs)}")

Indexed 500/52044
Indexed 1000/52044
Indexed 1500/52044
Indexed 2000/52044
Indexed 2500/52044
Indexed 3000/52044
Indexed 3500/52044
Indexed 4000/52044
Indexed 4500/52044
Indexed 5000/52044
Indexed 5500/52044
Indexed 6000/52044
Indexed 6500/52044
Indexed 7000/52044
Indexed 7500/52044
Indexed 8000/52044
Indexed 8500/52044
Indexed 9000/52044
Indexed 9500/52044
Indexed 10000/52044
Indexed 10500/52044
Indexed 11000/52044
Indexed 11500/52044
Indexed 12000/52044
Indexed 12500/52044
Indexed 13000/52044
Indexed 13500/52044
Indexed 14000/52044
Indexed 14500/52044
Indexed 15000/52044
Indexed 15500/52044
Indexed 16000/52044
Indexed 16500/52044
Indexed 17000/52044
Indexed 17500/52044
Indexed 18000/52044
Indexed 18500/52044
Indexed 19000/52044
Indexed 19500/52044
Indexed 20000/52044
Indexed 20500/52044
Indexed 21000/52044
Indexed 21500/52044
Indexed 22000/52044
Indexed 22500/52044
Indexed 23000/52044
Indexed 23500/52044
Indexed 24000/52044
Indexed 24500/52044
Indexed 25000/52044
Indexed 25500/52044
